In [1]:
import os
import pandas as pd
import numpy as np
from understatapi import UnderstatClient

In [2]:
# フォルダーの自動生成
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

In [3]:
TEAM_NAME = "Tottenham"
SEASONS = ["2022", "2023", "2024", "2025"]

In [4]:
# データの取得（2022-2025シーズン）
all_matches = []

with UnderstatClient() as understat:
    try:
        for season in SEASONS:
            matches = understat.team(team=TEAM_NAME).get_match_data(season=season)
            df_season = pd.DataFrame(matches)
            df_season["season"] = season
            all_matches.append(df_season)

    except Exception as e:
            print(f"Error fetching data for season {season}: {e}")

df_raw = pd.concat(all_matches, ignore_index=True)

# rawデータの保存
df_raw.to_csv("data/raw/spurs_raw_matches_2022_2025.csv", index=False)

In [5]:
# 前処理・データ加工
df = df_raw.copy()
df.shape

(152, 11)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        152 non-null    str   
 1   isResult  152 non-null    bool  
 2   side      152 non-null    str   
 3   h         152 non-null    object
 4   a         152 non-null    object
 5   goals     152 non-null    object
 6   xG        152 non-null    object
 7   datetime  152 non-null    str   
 8   forecast  152 non-null    object
 9   result    152 non-null    str   
 10  season    152 non-null    str   
dtypes: bool(1), object(5), str(5)
memory usage: 12.2+ KB


In [7]:
# チーム名抽出
df["home_team"] = df["h"].str["title"]
df["away_team"] = df["a"].str["title"]

# 得点数
df["home_goals"] = df["goals"].str["h"].astype(int)
df["away_goals"] = df["goals"].str["a"].astype(int)

# xG（ゴール期待値）
df["home_xG"] = df["xG"].str["h"].astype(float)
df["away_xG"] = df["xG"].str["a"].astype(float)

In [8]:
# スパーズ視点の指標算出
df["spurs_xG"] = np.where(df["home_team"] == "Tottenham", df["home_xG"], df["away_xG"])
df["spurs_xGA"] = np.where(df["home_team"] == "Tottenham", df["away_xG"], df["home_xG"])

In [9]:
df["spurs_goals"] = np.where(df["home_team"] == "Tottenham", df["home_goals"], df["away_goals"])
df["spurs_conceded"] = np.where(df["home_team"] == "Tottenham", df["away_goals"], df["home_goals"])

In [10]:
# スパーズ視点での勝敗(W/D/L)
conditions = [
    df["spurs_goals"] > df["spurs_conceded"],
    df["spurs_goals"] < df["spurs_conceded"]
]
choices = ["W", "L"]

df["spurs_result"] = np.select(conditions, choices, default="D")

In [11]:
# 日付処理
df["datetime"] = pd.to_datetime(df["datetime"])

In [12]:
# クリーニング済みのデータの選択と保存
cols_to_keep = [
    "season", "datetime", "home_team", "away_team",
    "home_goals", "away_goals", "spurs_goals", "spurs_conceded",
    "spurs_xG", "spurs_xGA", "spurs_result"
]

In [13]:
df_processed = df[cols_to_keep].copy()
df_processed.to_csv("data/processed/spurs_cleaned_match_data.csv", index=False)

In [14]:
df_processed.shape

(152, 11)

In [15]:
df_processed.head()

,season,datetime,home_team,away_team,home_goals,away_goals,spurs_goals,spurs_conceded,spurs_xG,spurs_xGA,spurs_result
0,2022,2022-08-06 14:00:00,Tottenham,Southampton,4,1,4,1,1.617200,0.386546,W
1,2022,2022-08-14 15:30:00,Chelsea,Tottenham,2,2,2,2,1.587080,1.839450,D
2,2022,2022-08-20 11:30:00,Tottenham,Wolverhampton Wanderers,1,0,1,0,1.466080,0.686123,W
3,2022,2022-08-28 15:30:00,Nottingham Forest,Tottenham,0,2,2,0,2.444730,0.801369,W
4,2022,2022-08-31 18:45:00,West Ham,Tottenham,1,1,1,1,0.723285,1.522670,D
